# 🏥 Hypertension RAG + Agents (Colab Setup)

This notebook sets up the entire Hypertension RAG and Agent system in Google Colab, ingests the ESC 2021 guidelines, runs the tests, and launches the Streamlit UI.

In [ ]:
# 1. Clone the repository and navigate to the hypertension project
!git clone https://github.com/Abdelrahmann-Mostafa/Pyramind---Hackathon.git
%cd Pyramind---Hackathon/hypertension_rag

# 2. Install dependencies (we relaxed pandas/numpy versions in requirements.txt)
!sed -i '/anthropic/d' requirements.txt
!pip install -r requirements.txt

## Setup Environment Variables
**IMPORTANT:** You need to provide your Groq API key here!

In [ ]:
import os
from google.colab import userdata

# Put your API key in Colab Secrets under 'GROQ_API_KEY' or paste it below
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
except:
    os.environ["GROQ_API_KEY"] = "your-api-key-here"  # REPLACE THIS IF NOT USING SECRETS

## Ingest Data & Run Tests

In [ ]:
# 3. Ingest the ESC 2021 Guidelines into ChromaDB
!python scripts/ingest_guidelines.py

# 4. Run the full test suite to ensure RAG and Agent reasoning work
!python tests/test_hypertension_rag.py

## Launch Streamlit App
We use `localtunnel` to expose the Streamlit port (8501) to the public web so you can interact with it.

In [ ]:
# 5. Install localtunnel globally so npx always finds it
!npm install -g localtunnel

# ------------------------------------------------------------------
# 6. FIX for the 502: start Streamlit DETACHED, WAIT until it is
#    healthy, and only THEN start localtunnel.
#    The old cell started both simultaneously (&) so the tunnel hit
#    port 8501 before Streamlit was listening, and the background
#    process was killed when the cell finished (SIGHUP) -> 502.
# ------------------------------------------------------------------
import os, socket, subprocess, time, urllib.request

if not os.environ.get("GROQ_API_KEY"):
    raise SystemExit("GROQ_API_KEY not set - add it to Colab Secrets (key icon) and re-run this cell.")

print("==============================================================")
print("COPY THIS IP ADDRESS. You will need it for the localtunnel page:")
print(urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip('\n'))
print("==============================================================")

def is_port_open(port, timeout=3.0):
    try:
        with socket.create_connection(('127.0.0.1', port), timeout=timeout):
            return True
    except OSError:
        return False

# (1) Start Streamlit as a DETACHED process (survives this cell ending)
log = open('streamlit.log', 'wb')
streamlit = subprocess.Popen(
    ['streamlit', 'run', 'app_hypertension.py',
     '--server.address', '0.0.0.0',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f'[1/3] Streamlit started (PID {streamlit.pid}). Waiting for port 8501 ...')

deadline = time.time() + 120
while time.time() < deadline:
    if is_port_open(8501):
        print('[1/3] OK - Port 8501 is open, Streamlit process is alive.')
        break
    if streamlit.poll() is not None:
        print('[1/3] ERROR - Streamlit exited early! Check streamlit.log below:\n')
        print(open('streamlit.log').read())
        raise SystemExit(1)
    time.sleep(2)
else:
    print('[1/3] ERROR - Port 8501 never opened. Check streamlit.log below:\n')
    print(open('streamlit.log').read())
    raise SystemExit(1)

# (2) Wait for the REAL health endpoint.
#     First run downloads/loads the ~400MB SentenceTransformer model,
#     which can take 1-3 minutes - during that time the tunnel would 502.
print('[2/3] Waiting for Streamlit health endpoint (first-run model load can take a few min) ...')
healthy = False
for _ in range(120):              # up to ~6 minutes
    try:
        with urllib.request.urlopen('http://127.0.0.1:8501/_stcore/health', timeout=3) as r:
            if r.read().decode().strip() == 'ok':
                healthy = True
                break
    except Exception:
        pass
    time.sleep(3)

if healthy:
    print('[2/3] OK - Streamlit is healthy and serving.')
else:
    print('[2/3] WARNING - Health endpoint not ready yet - app may still be loading heavy models.')
    print('     Tail streamlit.log for progress/errors:')
    print(open('streamlit.log').read()[-4000:])

# (3) NOW start localtunnel (only after Streamlit can serve)
print('[3/3] Starting localtunnel ...')
tunnel = subprocess.Popen(
    ['npx', 'localtunnel', '--port', '8501'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

# Stream the tunnel output so the public URL appears in real time
for raw in iter(tunnel.stdout.readline, b''):
    line = raw.decode(errors='replace').strip()
    if line:
        print(line, flush=True)
    if 'loca.lt' in line:
        url = [tok for tok in line.split() if tok.startswith('https://')]
        if url:
            print('\nOPEN THIS IN YOUR BROWSER: ' + url[-1])
            print('(Enter your Colab IP when localtunnel asks for the tunnel password)')
